In [18]:
from typing import Dict, TypedDict, Optional
from langgraph.graph import StateGraph, END
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain.vectorstores import FAISS

from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=gemini_api_key)
os.environ["GOOGLE_API_KEY"] = gemini_api_key

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

class GraphState(TypedDict):
    question: Optional[str] = None
    classification: Optional[str] = None
    response: Optional[str] = None
    length: Optional[int] = None
    greeting: Optional[str] = None

workflow = StateGraph(GraphState)


def retriever_qa_creation():
       pass
rag_chain = retriever_qa_creation()

from langchain_core.messages import HumanMessage

def classify(question):
    return llm([HumanMessage(content=f"classify intent of given input as greeting or not_greeting. Output just the class. Input: {question}")])

def classify_input_node(state):
    question = state.get('question', '').strip()

    classification = classify(question) 
    print("___________________________________________")
    print(classification)
    return {"classification": classification.content}

def handle_greeting_node(state):
    return {"greeting": "Hello! How can I help you today?"}

def handle_RAG(state):
    question = state.get('question', '').strip()
    prompt = question
    if state.get("length",0)<30:
         search_result = rag_chain.run(prompt)
    else:
         search_result = rag_chain.run(prompt+'. Return total count only.')

    return {"response": search_result,"length":len(search_result)}


def bye(state):
    return{"greeting":"The graph has finished"}

workflow.add_node("classify_input", classify_input_node)
workflow.add_node("handle_greeting", handle_greeting_node)
workflow.add_node("handle_RAG", handle_RAG)
workflow.add_node("bye", bye)

workflow.set_entry_point("classify_input")
workflow.add_edge('handle_greeting', END)
workflow.add_edge('bye', END)


def decide_next_node(state):
    return "handle_greeting" if state.get('classification') == "greeting" else "handle_RAG"

def check_RAG_length(state):
    print(state.get("length",0))
    if state.get("length",0)>30 :
        return "handle_RAG"
    else:
        return "bye"

workflow.add_conditional_edges(
    "classify_input",
    decide_next_node,
    {
        "handle_greeting": "handle_greeting",
        "handle_RAG": "handle_RAG"
    }
)

workflow.add_conditional_edges(
    "handle_RAG",
    check_RAG_length,
    {
        "bye": "bye",
        "handle_RAG": "handle_RAG"
    }
)

app = workflow.compile()
app.invoke({'question':'Hello bot','length':0})

___________________________________________
content='greeting' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run-81de7b2c-cdeb-4faa-ad0c-f255944f9a0b-0' usage_metadata={'input_tokens': 21, 'output_tokens': 2, 'total_tokens': 23, 'input_token_details': {'cache_read': 0}}


{'question': 'Hello bot',
 'classification': 'greeting',
 'length': 0,
 'greeting': 'Hello! How can I help you today?'}